# Notebook 01: BraTS Dataset Exploration & Pipeline Verification
**Project:** AI Based Brain Tumor Detection and Localization for Surgical Planning  
**Module:** Module 2 - BraTS Dataset Ingestion, Validation, Indexing, and DataLoader Pipeline  
**Hardware Target:** CPU (Local Laptop with Intel Iris Xe Graphics)  

### Objectives:
1. Discover and validate multi-modal MRI scans (`flair`, `t1`, `t1ce`, `t2`) and ground truth labels (`seg`).
2. Inspect voxel dimensions, physical spacing $(dx, dy, dz)$, and 4x4 affine matrices.
3. Generate structured CSV/JSON indexes and compute cohort statistics.
4. Perform reproducible, patient-level train/validation/test splitting.
5. Instantiate `BraTSDataset` and examine multi-modal channel stacking and metadata preservation.

In [ ]:
import sys
from pathlib import Path
import os

# Add repository root to Python path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f"Project root resolved: {ROOT_DIR}")

## 1. Configure Dataset Settings
Configure dataset paths using `DatasetConfig`. By default, this reads from `data/raw` or the `BRATS_DATA_ROOT` environment variable.

In [ ]:
from ai.config.dataset_config import DatasetConfig, DEFAULT_MODALITIES, LABEL_MAPPINGS, TARGET_REGIONS

config = DatasetConfig(
    dataset_name="BraTS2021",
    seed=42,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

print("Dataset Configuration:")
print(f" - Data Root       : {config.data_root}")
print(f" - Modalities      : {config.modalities}")
print(f" - Ground Truth    : {config.label_modality}")
print(f" - Label Mappings  : {LABEL_MAPPINGS}")
print(f" - Target Regions  : {TARGET_REGIONS}")

## 2. Scan and Validate Dataset Cases
Scan the dataset directory, verify that required modalities exist, check that all modalities share matching dimensions, and confirm files are not corrupt.

In [ ]:
from ai.data.dataset_indexer import BraTSDatasetIndexer
from ai.data.statistics import BraTSDatasetStatistics

if not config.data_root.exists() or not any(config.data_root.iterdir()):
    print(f"[!] Notice: No cases currently in {config.data_root}.")
    print("    Place authentic BraTS cases in data/raw or set BRATS_DATA_ROOT to your downloaded directory.")
else:
    indexer = BraTSDatasetIndexer(config)
    records, validation_summary = indexer.build_index()
    
    stats = BraTSDatasetStatistics.compute_summary(records, validation_summary)
    print(BraTSDatasetStatistics.format_text_report(stats))
    
    # Export structured index files
    csv_path = config.index_output_dir / "brats_index.csv"
    indexer.export_csv(records, csv_path)
    print(f"[+] Index saved to: {csv_path}")

## 3. Patient-Level Splitting (Leakage-Free)
Partition patient cases into train (70%), validation (15%), and test (15%) splits. All slices from each patient remain strictly within the assigned partition.

In [ ]:
from ai.data.split import PatientLevelSplitter

splitter = PatientLevelSplitter(config)

# Example with mock or real case IDs
sample_case_ids = [f"BraTS2021_{i:05d}" for i in range(1, 51)]
split = splitter.split_cases(sample_case_ids, seed=42)

print(f"Total Cases: {split.total_cases}")
print(f" - Train set : {len(split.train_cases)} cases")
print(f" - Val set   : {len(split.val_cases)} cases")
print(f" - Test set  : {len(split.test_cases)} cases")

# Verify zero overlap
assert set(split.train_cases).isdisjoint(set(split.val_cases))
assert set(split.train_cases).isdisjoint(set(split.test_cases))
print("[+] Data leakage check passed: 100% disjoint patient partitions.")

## 4. PyTorch Dataset Inspection
Instantiate `BraTSDataset` and inspect the item dictionary to confirm multi-channel stacking and spatial metadata preservation.

In [ ]:
from ai.data.brats_dataset import BraTSDataset

# When index records exist, instantiate dataset:
# dataset = BraTSDataset(case_records=records, load_data=True)
# item = dataset[0]
# print("Item keys:", item.keys())
# print("Image shape:", item['image'].shape) # Expected: [4, D, H, W]
# print("Voxel spacing (mm):", item['voxel_spacing'])
# print("Affine Matrix:", item['affine'])
print("BraTSDataset ready for DataLoader wrapping in training scripts.")

## 5. Multi-Planar Slice Visualization
Use `BraTSVisualizer` to explore authentic 2D cross-sections (axial, coronal, sagittal) with semi-transparent segmentation overlays.

In [ ]:
from ai.data.visualizer import BraTSVisualizer

visualizer = BraTSVisualizer()
print("Visualizer loaded. Call visualizer.render_and_save_comparison(...) on real cases.")